In [ ]:
import numpy as np #handles numerical operations
import pandas as pd #used to work with tables/datasets

import io #helps with input/output operations

from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
#LabelEncoder converts labels into numbers(binary)
#StandardScaler scales using the formula (x-mu)/sigma
#RobustScaler is used when outliers are present, it uses median to scale the data

from sklearn.feature_selection import VarianceThreshold #removes features that barely change.

from sklearn.model_selection import train_test_split
#Training data teaches the model and Testing data checks how well the model performs

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
#Accuracy is correct predictions over total predictions
#Confusion matrix: [TP,TN;FP,FN]
#Classification report has: precision, recall, F1-score, support

from sklearn.svm import SVC
#Support Vector Machine tries to find the best boundary separating classes.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00254/biodeg.csv"

columns = [
    'SpMax_L', 'J_Dz_e', 'nHM', 'F01_N', 'F04_C',
    'NssssC', 'nCb', 'C', 'nCp', 'nO',
    'F03_N', 'SdssC', 'HyWi_B', 'LOC', 'SM6_L',
    'F03_C', 'Me', 'Mi', 'nN', 'nArNO2',
    'nCRX3', 'SpPosA_B', 'nCIR', 'B01_C', 'B03_C',
    'N', 'SpMax_A', 'Psi_i_1d', 'B04_C', 'SdO',
    'TI2_L', 'nCrt', 'Crt', 'Is_biodegradable'
]
#Columns are descriptors

data = pd.read_csv(url, sep=';', header=None, names=columns)

data.head(1055)

In [ ]:
# Remove duplicate rows
data = data.drop_duplicates()

# Remove missing values
data = data.dropna()

# Split features and target
X = data.drop('Is_biodegradable', axis=1) #Contains all input features
y = data['Is_biodegradable'] #Contains target labels

encoder = LabelEncoder()

y = encoder.fit_transform(y)


selector = VarianceThreshold(threshold=0.01) #Removes low variance columns

X_var = selector.fit_transform(X) #Applies variance filtering

selected_columns = X.columns[selector.get_support()] #Gets names of the remaining useful columns

X = pd.DataFrame(X_var, columns=selected_columns) #Converts the data back into the dataset


corr_matrix = X.corr().abs() #If two features contain almost identical information, we remove one

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.95)
]

X = X.drop(columns=to_drop)

 
scaler = RobustScaler() #Scale features

X_scaled = scaler.fit_transform(X)

print("Final shape:", X_scaled.shape)

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)


# Create Logistic Regression model
model = LogisticRegression(max_iter=5000)


# Train the model
model.fit(X_train, y_train)


# Make predictions
y_pred = model.predict(X_test)


# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Non-Biodegradable', 'Biodegradable'],
    yticklabels=['Non-Biodegradable', 'Biodegradable']
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()


# Detailed Classification Metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

#Precision=TP/(TP+FP)how many truly were biodegradable?
#Recall=TP/(TP+FN) how many did the model successfully detect
#F1-score balances precision and recall. F1=2(Precision*Recall)/(Precision+Recall)
#Support is the number of true occurrences of each class.

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)


# Create Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

#Creates 100 decision trees. Each tree votes. Final prediction is based on majority voting.


# Train the model
model.fit(X_train, y_train)


# Make predictions
y_pred = model.predict(X_test)


# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Reds',
    xticklabels=['Non-Biodegradable', 'Biodegradable'],
    yticklabels=['Non-Biodegradable', 'Biodegradable']
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()


# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Create SVM model
model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)
#RBF = Radial Basis Function. Allows non-linear classification. This is important as chemical data is rarely linearly separable


# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)


# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Purples',
    xticklabels=['Non-Biodegradable', 'Biodegradable'],
    yticklabels=['Non-Biodegradable', 'Biodegradable']
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
#Overall workflow
#Load Dataset ---> Clean Data ---> Encode Labels --->  Feature Selection ---> Scale Features ---> Train/Test Split ---> Train Models --->Evaluate Performance